# NB_bishop_ch20_figures
## Key Figures for Bishop Chapter 20 -- Diffusion Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_20/NB_bishop_ch20_figures.ipynb)

Generate publication-quality figures for forward diffusion, noise schedules, reverse denoising, and U-Net architecture.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

COLORS = {'blue': '#1A3A6E', 'red': '#CD0000', 'green': '#2E7D32', 'amber': '#B5853F'}

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            legend._ncols = min(len(legend.get_texts()), 4)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

np.random.seed(42)

## Figure 20.1 -- Forward Diffusion Process

A simple pattern progressively corrupted with noise at t=0, T/4, T/2, 3T/4, T.

In [ ]:
np.random.seed(42)
img_size = 32

# Create a clean structured image (concentric rings)
y_coords, x_coords = np.ogrid[-img_size//2:img_size//2, -img_size//2:img_size//2]
r = np.sqrt(x_coords**2 + y_coords**2)
clean_image = np.sin(r * 0.8) * 0.5 + 0.5

# Linear noise schedule
T = 1000
beta = np.linspace(1e-4, 0.02, T)
alpha = 1 - beta
alpha_bar = np.cumprod(alpha)

timesteps = [0, T//4, T//2, 3*T//4, T-1]
labels = ['$t = 0$', '$t = T/4$', '$t = T/2$', '$t = 3T/4$', '$t = T$']

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))

for i, (t, label) in enumerate(zip(timesteps, labels)):
    if t == 0:
        noisy = clean_image.copy()
    else:
        ab = alpha_bar[t]
        noise = np.random.randn(img_size, img_size)
        noisy = np.sqrt(ab) * clean_image + np.sqrt(1 - ab) * noise

    axes[i].imshow(noisy, cmap='Blues', interpolation='nearest')
    axes[i].set_title(label, fontsize=13, fontweight='bold')
    axes[i].axis('off')

    # Draw forward arrow between panels
    if i < 4:
        fig.text((i + 1) / 5 - 0.01, 0.5, '$\\rightarrow$', fontsize=20,
                 ha='center', va='center', color=COLORS['red'], transform=fig.transFigure)

fig.suptitle('Forward Diffusion: Progressively Adding Noise', fontweight='bold', fontsize=14, y=1.05)
fig.tight_layout()
save_fig(fig, 'fig20_1_forward_diffusion')
plt.show()

## Figure 20 -- Noise Schedule

Plot $\beta_t$, $\alpha_t$, and $\bar{\alpha}_t$ vs timestep $t$ for a linear schedule.

In [ ]:
T = 1000
t_range = np.arange(T)
beta_t = np.linspace(1e-4, 0.02, T)
alpha_t = 1 - beta_t
alpha_bar_t = np.cumprod(alpha_t)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: beta_t
ax1.plot(t_range, beta_t, color=COLORS['red'], linewidth=2.5, label='$\\beta_t$')
ax1.set_xlabel('Timestep $t$', fontsize=12)
ax1.set_ylabel('$\\beta_t$', fontsize=13, color=COLORS['red'])
ax1.set_title('Noise Variance Schedule', fontweight='bold', fontsize=13)
ax1.legend(fontsize=12)

# Right: alpha_t and alpha_bar_t
ax2.plot(t_range, alpha_t, color=COLORS['blue'], linewidth=2.5, label='$\\alpha_t = 1 - \\beta_t$')
ax2.plot(t_range, alpha_bar_t, color=COLORS['green'], linewidth=2.5,
         label='$\\bar{\\alpha}_t = \\prod_{s=1}^{t} \\alpha_s$')
ax2.axhline(y=0.5, color=COLORS['amber'], linestyle='--', linewidth=1, alpha=0.7)
ax2.set_xlabel('Timestep $t$', fontsize=12)
ax2.set_ylabel('Value', fontsize=13)
ax2.set_title('Signal Retention over Time', fontweight='bold', fontsize=13)
ax2.legend(fontsize=11)

# Annotate key region
half_idx = np.argmin(np.abs(alpha_bar_t - 0.5))
ax2.annotate(f'$\\bar{{\\alpha}}_t = 0.5$ at $t \\approx {half_idx}$',
             xy=(half_idx, 0.5), xytext=(half_idx + 150, 0.65),
             fontsize=10, arrowprops=dict(arrowstyle='->', color=COLORS['amber'], lw=1.5),
             color=COLORS['amber'])

fig.tight_layout()
save_fig(fig, 'fig20_noise_schedule')
plt.show()

## Figure 20 -- Reverse Denoising Process

Noise -> progressively cleaner images (mirror of forward diffusion).

In [ ]:
np.random.seed(42)
img_size = 32

# Same clean image
y_coords, x_coords = np.ogrid[-img_size//2:img_size//2, -img_size//2:img_size//2]
r = np.sqrt(x_coords**2 + y_coords**2)
clean_image = np.sin(r * 0.8) * 0.5 + 0.5

# Reverse: show t=T, 3T/4, T/2, T/4, 0
T = 1000
beta = np.linspace(1e-4, 0.02, T)
alpha = 1 - beta
alpha_bar = np.cumprod(alpha)

reverse_timesteps = [T-1, 3*T//4, T//2, T//4, 0]
reverse_labels = ['$t = T$\n(noise)', '$t = 3T/4$', '$t = T/2$', '$t = T/4$', '$t = 0$\n(clean)']

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))

for i, (t, label) in enumerate(zip(reverse_timesteps, reverse_labels)):
    if t == 0:
        img = clean_image.copy()
    else:
        ab = alpha_bar[t]
        noise = np.random.randn(img_size, img_size)
        img = np.sqrt(ab) * clean_image + np.sqrt(1 - ab) * noise

    axes[i].imshow(img, cmap='Greens', interpolation='nearest')
    axes[i].set_title(label, fontsize=12, fontweight='bold')
    axes[i].axis('off')

    if i < 4:
        fig.text((i + 1) / 5 - 0.01, 0.5, '$\\rightarrow$', fontsize=20,
                 ha='center', va='center', color=COLORS['green'], transform=fig.transFigure)

fig.suptitle('Reverse Denoising: Progressively Removing Noise', fontweight='bold', fontsize=14, y=1.05)
fig.tight_layout()
save_fig(fig, 'fig20_reverse_process')
plt.show()

## Figure 20 -- Simplified U-Net Architecture

Encoder (downsampling) + decoder (upsampling) with skip connections.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(-1, 15)
ax.set_ylim(-1, 10)
ax.axis('off')

def draw_ublock(ax, x, y, w, h, label, fc, ec, fs=9):
    box = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                                   facecolor=fc, edgecolor=ec, linewidth=2, alpha=0.85)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=fs, fontweight='bold', color=ec)

bw = 1.8  # block width

# Encoder (left side, going down)
enc_configs = [
    (0.5, 7.5, 'Conv 64\n+ ReLU', 1.2),
    (0.5, 5.5, 'Conv 128\n+ ReLU', 1.2),
    (0.5, 3.5, 'Conv 256\n+ ReLU', 1.2),
]

for x, y, label, h in enc_configs:
    draw_ublock(ax, x, y, bw, h, label, '#D6E4F0', COLORS['blue'], 9)

# Down arrows
for i in range(len(enc_configs) - 1):
    _, y1, _, h1 = enc_configs[i]
    _, y2, _, _ = enc_configs[i + 1]
    ax.annotate('', xy=(0.5 + bw/2, y2 + enc_configs[i+1][3]),
                xytext=(0.5 + bw/2, y1),
                arrowprops=dict(arrowstyle='->', color=COLORS['blue'], lw=2))
    ax.text(0.5 + bw + 0.2, (y1 + y2 + enc_configs[i+1][3]) / 2 + 0.2,
            'MaxPool', fontsize=8, color=COLORS['blue'], rotation=-90, ha='center')

# Bottleneck
draw_ublock(ax, 5.5, 1.5, 3, 1.2, 'Bottleneck\nConv 512 + ReLU\n+ Time Embedding',
            '#FFF3E0', COLORS['amber'], 9)

# Arrow encoder -> bottleneck
ax.annotate('', xy=(5.5, 2.1), xytext=(0.5 + bw, 4.1),
            arrowprops=dict(arrowstyle='->', color=COLORS['blue'], lw=2))
ax.text(3.5, 2.5, 'MaxPool', fontsize=8, color=COLORS['blue'], rotation=-30)

# Decoder (right side, going up)
dec_configs = [
    (11.5, 3.5, 'ConvT 256\n+ ReLU', 1.2),
    (11.5, 5.5, 'ConvT 128\n+ ReLU', 1.2),
    (11.5, 7.5, 'ConvT 64\n+ ReLU', 1.2),
]

for x, y, label, h in dec_configs:
    draw_ublock(ax, x, y, bw, h, label, '#FCE4EC', COLORS['red'], 9)

# Up arrows
for i in range(len(dec_configs) - 1):
    _, y1, _, h1 = dec_configs[i]
    _, y2, _, _ = dec_configs[i + 1]
    ax.annotate('', xy=(11.5 + bw/2, y2),
                xytext=(11.5 + bw/2, y1 + h1),
                arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=2))
    ax.text(11.5 - 0.3, (y1 + h1 + y2) / 2,
            'Upsample', fontsize=8, color=COLORS['red'], rotation=90, ha='center')

# Arrow bottleneck -> decoder
ax.annotate('', xy=(11.5, 4.1), xytext=(8.5, 2.1),
            arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=2))
ax.text(10.3, 2.5, 'Upsample', fontsize=8, color=COLORS['red'], rotation=30)

# Skip connections
skip_pairs = [(0, 2), (1, 1), (2, 0)]  # encoder idx -> decoder idx
for enc_idx, dec_idx in skip_pairs:
    _, ey, _, eh = enc_configs[enc_idx]
    _, dy, _, dh = dec_configs[dec_idx]
    y_mid = ey + eh / 2
    ax.annotate('', xy=(11.5, dy + dh / 2),
                xytext=(0.5 + bw, y_mid),
                arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=1.8,
                                linestyle='--', connectionstyle='arc3,rad=-0.05'))

ax.text(7, 8.7, 'Skip Connections', fontsize=11, ha='center', fontweight='bold',
        color=COLORS['green'], fontstyle='italic')

# Input / Output labels
ax.text(0.5 + bw/2, 9.2, 'Input\n$x_t$ + $t$', ha='center', fontsize=10,
        fontweight='bold', color='#333333')
ax.annotate('', xy=(0.5 + bw/2, 8.7), xytext=(0.5 + bw/2, 9.0),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

ax.text(11.5 + bw/2, 9.2, 'Output\n$\\hat{\\epsilon}$', ha='center', fontsize=10,
        fontweight='bold', color='#333333')
ax.annotate('', xy=(11.5 + bw/2, 9.0), xytext=(11.5 + bw/2, 8.7),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# Legend
enc_patch = mpatches.Patch(facecolor='#D6E4F0', edgecolor=COLORS['blue'], label='Encoder (downsample)')
dec_patch = mpatches.Patch(facecolor='#FCE4EC', edgecolor=COLORS['red'], label='Decoder (upsample)')
bot_patch = mpatches.Patch(facecolor='#FFF3E0', edgecolor=COLORS['amber'], label='Bottleneck')
skip_line = plt.Line2D([0], [0], color=COLORS['green'], lw=1.8, ls='--', label='Skip connection')
ax.legend(handles=[enc_patch, dec_patch, bot_patch, skip_line],
          loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=4, fontsize=10, framealpha=0.0)

ax.set_title('U-Net Architecture for Diffusion Models', fontweight='bold', fontsize=15, pad=15)

fig.tight_layout()
save_fig(fig, 'fig20_unet')
plt.show()

## Summary

- **fig20_1**: Forward diffusion showing progressive noise addition
- **fig20_noise_schedule**: Linear noise schedule with $\beta_t$, $\alpha_t$, $\bar{\alpha}_t$
- **fig20_reverse_process**: Reverse denoising from pure noise to clean image
- **fig20_unet**: U-Net architecture with encoder, decoder, and skip connections